In [32]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", "split")

In [33]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [34]:
df_train=ds['train'].to_pandas()
df_test=ds['test'].to_pandas()
df_validation=ds['validation'].to_pandas()

In [35]:
df_train.shape

(16000, 2)

In [36]:
df_test.shape

(2000, 2)

In [37]:
df_validation.shape

(2000, 2)

In [38]:
df_train.isnull().sum()

text     0
label    0
dtype: int64

In [39]:
df_train.duplicated().sum()

np.int64(1)

In [40]:
df_train['label'].unique()

array([0, 3, 2, 5, 4, 1])

In [41]:
ds['train'].features

{'text': Value('string'),
 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}

In [42]:
def mapper(label):
   if label in [0,4]:
       return "High"
   elif label == 3:
       return "Medium"
   else :
        return "Low"
    

In [43]:
df_train['stress_level']=df_train['label'].apply(mapper)
df_test['stress_level']=df_test['label'].apply(mapper)
df_validation['stress_level']=df_validation['label'].apply(mapper)

In [44]:
df_train

,text,label,stress_level
0,i didnt feel humiliated,0,High
1,i can go from feeling so hopeless to so damned...,0,High
2,im grabbing a minute to post i feel greedy wrong,3,Medium
3,i am ever feeling nostalgic about the fireplac...,2,Low
4,i am feeling grouchy,3,Medium
...,...,...,...
15995,i just had a very brief time in the beanbag an...,0,High
15996,i am now turning and i feel pathetic that i am...,0,High
15997,i feel strong and good overall,1,Low
15998,i feel like this was such a rude comment and i...,3,Medium


In [45]:
import re
def text_cleaner(text):
    text=text.lower()
    # text = re.sub(r'[^a-zA-Z ]', '', text)       # remove special chars
    # text = re.sub(r'\s+', ' ', text).strip()  
    return text

In [46]:
df_train['clean_text']=df_train['text'].apply(text_cleaner)
df_test['clean_text']=df_test['text'].apply(text_cleaner)
df_validation['clean_text']=df_validation['text'].apply(text_cleaner)

In [47]:
df_train = df_train[~df_train['label'].isin([2,5])]
df_validation = df_validation[~df_validation['label'].isin([2,5])]
df_test = df_test[~df_test['label'].isin([2,5])]



In [48]:
df_train = df_train.reset_index(drop=True)
df_validation = df_validation.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [49]:
df_train

,text,label,stress_level,clean_text
0,i didnt feel humiliated,0,High,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...,0,High,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong,3,Medium,im grabbing a minute to post i feel greedy wrong
3,i am feeling grouchy,3,Medium,i am feeling grouchy
4,ive been feeling a little burdened lately wasn...,0,High,ive been feeling a little burdened lately wasn...
...,...,...,...,...
14119,i just had a very brief time in the beanbag an...,0,High,i just had a very brief time in the beanbag an...
14120,i am now turning and i feel pathetic that i am...,0,High,i am now turning and i feel pathetic that i am...
14121,i feel strong and good overall,1,Low,i feel strong and good overall
14122,i feel like this was such a rude comment and i...,3,Medium,i feel like this was such a rude comment and i...


In [50]:
y_train=df_train['label']
y_test=df_test['label']
y_val=df_validation['label']

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [52]:
vectorizer=TfidfVectorizer(max_features=10000,ngram_range=(1,2))

In [53]:
x_train=vectorizer.fit_transform(df_train['clean_text'])

In [54]:
x_test=vectorizer.transform(df_test['clean_text'])
x_validation=vectorizer.transform(df_validation['clean_text'])

In [55]:
x_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 309385 stored elements and shape (14124, 10000)>

In [56]:
from sklearn.linear_model import LogisticRegression

Logistic_model=LogisticRegression(class_weight='balanced')
Logistic_model.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [57]:
from sklearn.ensemble import RandomForestClassifier
randdom_forest_model = RandomForestClassifier()

randdom_forest_model.fit(x_train,y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [58]:
from sklearn.naive_bayes import MultinomialNB
multi=MultinomialNB(alpha=0.5)
multi.fit(x_train,y_train)

,alpha,0.5
,force_alpha,True
,fit_prior,True
,class_prior,None


In [59]:
from sklearn.metrics import accuracy_score

In [60]:
logistic_pred=Logistic_model.predict(x_validation)

In [61]:
# accuracy_score(logistic_pred,y_test)   # accuracy score of logistic model

In [62]:
forest_predic=randdom_forest_model.predict(x_validation)  # accuracy score for random forest 

In [63]:
accuracy_score(forest_predic,y_test)

ValueError: Found input variables with inconsistent numbers of samples: [1741, 1775]

In [ ]:
multinominial_pred=multi.predict(x_validation) 

In [ ]:
accuracy_score(multinominial_pred,y_test)  # accuracy score for naive byes

In [ ]:
df_train['clean_text'].sample(10)

In [ ]:
x_train.shape

In [ ]:
y_train.unique()

In [ ]:
import numpy as np
np.unique(multinominial_pred, return_counts=True)

In [ ]:
x_train.shape

In [ ]:
x_validation.shape

In [ ]:
x_test.shape